# Phase 4: Architecture Decision Matrix
Reasoning-only notebook for scenario-based embedding architecture choices.

## Scenario A - Offline Mobile
Recommended: HuggingFace all-MiniLM-L6-v2.
Rationale: On-device inference, no network dependency, compact vectors.
Trade-offs: Lower quality than larger cloud models and slower model updates.

## Scenario B - E-Commerce 500M
Recommended: OpenAI text-embedding-3-small + binary quantization.
Rationale: Strong quality and practical memory at very large scale.
Trade-offs: Quantization introduces quality loss and index refresh complexity.

## Scenario C - Legal Discovery
Recommended: Voyage legal embeddings + Cohere rerank.
Rationale: Domain precision and strong final ranking quality.
Trade-offs: Higher latency and higher operating cost.

## Scenario D - Multilingual SaaS
Recommended: OpenAI text-embedding-3-large.
Rationale: Unified multilingual space with minimal per-language ops overhead.
Trade-offs: Expensive storage and higher API cost.

## Scenario E - Multimodal
Recommended: CLIP-family architecture for text-image shared space.
Rationale: Supports cross-modal retrieval workflows.
Trade-offs: Added image pipeline complexity and weaker modality-specific depth.

## Cross-Scenario Summary
| Scenario | Provider | Model | Dims | Compression | Reranking |
|---|---|---|---|---|---|
| Offline Mobile | HuggingFace | all-MiniLM-L6-v2 | 384 | None | None |
| E-Commerce 500M | OpenAI | text-embedding-3-small | 1536 -> 192B | Binary | None |
| Legal Discovery | Voyage AI + Cohere | voyage-law-2 + rerank-english-v3.0 | 1024 | None | Yes |
| Multilingual SaaS | OpenAI | text-embedding-3-large | 3072 | Matryoshka 256 | Optional |
| Multimodal | OpenAI/CLIP | CLIP ViT-L/14 | 768 | None | None |

## Assignment Questions

### Q1: Which embedding provider/model would you choose for each scenario and why?

- **A (Offline Mobile):** `all-MiniLM-L6-v2` — the only scenario where no API is viable. At 22M parameters it runs on CPU, exports to ONNX/CoreML for Android/iOS, and needs no network. Accuracy loss versus cloud models is acceptable when the alternative is no search at all.

- **B (E-Commerce 500M):** `text-embedding-3-small` — general-purpose is sufficient for product search; 1536 dims before binary quantization keeps final storage to ~96 GB for 500M docs versus ~3 TB for float32. The `3-large` model would double storage without a meaningful accuracy gain on product descriptions.

- **C (Legal Discovery):** `voyage-law-2` — a general model trained on internet text cannot reliably distinguish *landlord not liable* from *tenant not liable* because both sit in similar liability-related neighborhoods in a generalist latent space. `voyage-law-2` is trained on legal corpora and understands that legal role (landlord vs tenant) is a primary semantic axis, not a surface variation.

- **D (Multilingual SaaS):** `text-embedding-3-large` — the only single model covering 100+ languages in a shared semantic space. A French query and an English document produce geometrically close vectors without any translation step. Running per-language models would require separate pipelines, indexes, and maintenance cycles — incompatible with the minimal ML infrastructure constraint.

- **E (Multimodal):** CLIP `ViT-L/14` — the only architecture that projects text and images into a shared 768-dim embedding space via contrastive training. A text query and an image of the same concept produce similar vectors, enabling cross-modal retrieval. No text-only model can do this.

---

### Q2: What trade-offs influenced your decisions (cost, accuracy, latency, infrastructure)?

| Scenario | Cost | Accuracy | Latency | Infrastructure |
|:---|:---|:---|:---|:---|
| A: Offline Mobile | Zero (local) | ~85–90% vs cloud | Fastest (on-device) | Minimal — bundled in app |
| B: E-Commerce | Medium (500M API calls, one-time) | ~90–95% of float32 | Fast (Hamming search) | Medium — FAISS cluster |
| C: Legal Discovery | High (Voyage AI + Cohere per query) | Highest (domain + reranker) | High (~400–500ms) | High — two-stage pipeline |
| D: Multilingual SaaS | High (`3-large` API rate) | High on major languages | Medium | Low — single model/index |
| E: Multimodal | Low (CLIP is open-source) | Medium (shallower per modality) | Medium | Medium — image pipeline |

The dominant trade-off in every scenario is **accuracy vs operational constraint**:
- B accepts ~5–10% accuracy loss to make 500M docs storable.
- C pays 5–10× higher cost and latency to guarantee precision where errors have professional consequences.
- D accepts marginal accuracy loss on low-resource languages to eliminate per-language ops overhead.

---

### Q3: How does a multimodal embedding space differ from a text-only embedding space?

A **text-only embedding space** (e.g., `text-embedding-3-large`) dedicates its full model capacity to linguistic understanding — negations, domain vocabulary, syntactic structure, semantic entailment. It cannot accept images as input.

A **multimodal embedding space** (e.g., CLIP) must satisfy a harder objective: align representations from two completely different input modalities so that paired examples (photo + caption) are geometrically close, while unpaired examples are far apart. This contrastive training creates a shared space where the same distance function works across modalities — but the trade-off is that neither modality receives the model's full representational depth.

Concretely:
- CLIP's text encoder is weaker on complex syntax and negations than `text-embedding-3-large`.
- CLIP's image encoder is weaker on fine-grained visual categories than a vision-specialized model.
- But CLIP can retrieve images given a text query, or retrieve text given an image query — something a text-only model fundamentally cannot do.

**Architectural implication for Scenario E:** If high-precision text-to-text retrieval is also required alongside image search, a hybrid architecture using both a text-only model (for text→text) and CLIP (for text→image, image→*) may be preferable — at the cost of maintaining two indexes and a query-routing layer.